In [ ]:
# ============================================================
# Block 1: Read + Process(by var) + Save cache
# ============================================================
import os
import glob
from collections import Counter

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt

from matplotlib.patches import Patch
from matplotlib.lines import Line2D
from matplotlib import rcParams
from dask.diagnostics import ProgressBar

# ---------------- Paths ----------------
RAW_ROOT = "/mnt/soclim0/public_data/weiji"
OUT_ROOT = "/home/weiji/restart_exam/code/20260226B2000WCN_BWCN_MERRA2_COMPARE/RESULT"

DATA_ROOT = os.path.join(OUT_ROOT, "data")
RANK_ROOT = os.path.join(OUT_ROOT, "rankings")
PLOT_ROOT = os.path.join(OUT_ROOT, "plots")

for p in [OUT_ROOT, DATA_ROOT, RANK_ROOT, PLOT_ROOT]:
    os.makedirs(p, exist_ok=True)

# ---------------- Experiments ----------------
EXP_CFG = {
    "BWCN": {
        "U_dir": os.path.join(RAW_ROOT, "BWCN", "U"),
        "U_pattern": "BWCN.cam.h3.*.U.nc",
        "T_dir": os.path.join(RAW_ROOT, "BWCN", "T"),
        "T_pattern": "BWCN.cam.h3.*.T.nc",
        "O3_dir": os.path.join(RAW_ROOT, "BWCN", "O3"),
        "O3_pattern": "BWCN.cam.h3.*.O3.nc",
    },
    "B2000WCN": {
        "U_dir": os.path.join(RAW_ROOT, "B2000WCN001002", "U"),
        "U_pattern": "B2000WCN.sample.cam.h3.*.U.nc",
        "T_dir": os.path.join(RAW_ROOT, "B2000WCN001002", "T"),
        "T_pattern": "B2000WCN.sample.cam.h3.*.T.nc",
        "O3_dir": os.path.join(RAW_ROOT, "B2000WCN001002", "O3"),
        "O3_pattern": "B2000WCN.sample.cam.h3.*.O3.nc",
    }
}

# ---------------- Constants ----------------
O3_PTOP = 30
O3_PBOT = 70
TOP_N = 5
LOW_FRAC = 0.25

TOP_COLORS = ['#1f77b4', '#d62728', '#9467bd', '#17becf', '#2ca02c']

rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "DejaVu Sans"],
    "font.size": 9,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.linewidth": 0.8,
    "xtick.direction": "out",
    "ytick.direction": "out",
})


# ============================================================
# File / coordinate helpers
# ============================================================
def parse_file_year(path):
    return int(os.path.basename(path).split(".")[-3])


def open_daily_files(exp, data_dir, pattern, chunks=None):
    """
    Unified reader:
      - BWCN: direct open_mfdataset
      - B2000WCN: split Run001 / Run002 and shift Run002 by +103 years
    """
    files = sorted(glob.glob(os.path.join(data_dir, pattern)))
    if not files:
        raise FileNotFoundError(f"[{exp}] No files found: {os.path.join(data_dir, pattern)}")

    if exp == "B2000WCN":
        files_001 = [f for f in files if 1 <= parse_file_year(f) <= 103]
        files_002 = [f for f in files if 105 <= parse_file_year(f) <= 210]

        if len(files_001) == 0 or len(files_002) == 0:
            raise RuntimeError(f"[{exp}] Run001 / Run002 split failed.")

        print(f"[{exp}] Loading Run001: {len(files_001)} files")
        ds1 = xr.open_mfdataset(files_001, combine="by_coords", chunks=chunks)

        print(f"[{exp}] Loading Run002: {len(files_002)} files (shift +103 years)")
        ds2 = xr.open_mfdataset(files_002, combine="by_coords", chunks=chunks)

        new_times = [t.replace(year=t.year + 103) for t in ds2.time.values]
        ds2 = ds2.assign_coords(time=new_times)

        ds = xr.concat([ds1, ds2], dim="time").sortby("time")
    else:
        print(f"[{exp}] Loading {len(files)} files")
        ds = xr.open_mfdataset(files, combine="by_coords", chunks=chunks).sortby("time")

    return ds


def find_lev_name(da):
    for name in ("lev", "plev", "lev_p", "level"):
        if name in da.dims or name in da.coords:
            return name
    raise ValueError(f"Cannot find vertical coordinate in dims={da.dims}")


def normalize_lev_to_pa(da):
    lev_name = find_lev_name(da)
    lev_vals = da[lev_name].values
    lev_pa = lev_vals * 100.0 if float(np.nanmax(lev_vals)) <= 2000.0 else lev_vals

    if lev_name != "lev":
        da = da.rename({lev_name: "lev"})
    da = da.assign_coords(lev=("lev", lev_pa))
    da["lev"].attrs["units"] = "Pa"
    return da


def select_level_da(da_2d, target_hpa):
    """
    Select nearest target level from DataArray(time, lev).
    """
    target_pa = target_hpa * 100.0
    idx = int(np.argmin(np.abs(da_2d["lev"].values - target_pa)))
    actual_hpa = float(da_2d["lev"].values[idx] / 100.0)
    return da_2d.isel(lev=idx), actual_hpa


# ============================================================
# Variable-specific processors
# ============================================================
def calc_U60N(ds, var="U"):
    """
    U -> zonal mean over lon -> nearest 60N
    Returns: U_60N(time, lev)
    """
    da = ds[var]

    if "lon" in da.dims:
        da = da.mean("lon")

    if "lat" not in da.dims:
        raise ValueError(f"{var} has no 'lat' dimension")

    actual_lat = float(da["lat"].sel(lat=60.0, method="nearest").values)
    da = da.sel(lat=60.0, method="nearest")
    da = normalize_lev_to_pa(da)

    print(f"[calc_U60N] Requested 60N, actual grid latitude = {actual_lat:.2f}°")
    return da


def calc_minT_pcap(ds, var="T"):
    """
    T -> zonal mean over lon -> 60-90N min over lat
    Returns: T_min_60_90N(time, lev)
    """
    da = ds[var]

    if "lon" in da.dims:
        da = da.mean("lon")

    if "lat" not in da.dims:
        raise ValueError(f"{var} has no 'lat' dimension, dims={da.dims}")

    da_cap = da.sel(lat=slice(60, 90))
    da_min = da_cap.min(dim="lat")

    da_min = normalize_lev_to_pa(da_min)
    return da_min


def calc_parc_o3_hybrid(ds_sub, p_top_hpa, p_bot_hpa):
    """
    Hybrid-coordinate O3 column integration (DU) over [p_top_hpa, p_bot_hpa].
    """
    g = 9.80665
    M_air = 28.964 / 1000.0
    Na = 6.02214e23
    factor = Na / (g * M_air * 2.687e20)

    P0 = ds_sub["P0"]
    PS = ds_sub["PS"]

    P_interface = ds_sub["hyai"] * P0 + ds_sub["hybi"] * PS

    p_i = (
        P_interface.isel(ilev=slice(0, -1))
        .rename({"ilev": "lev"})
        .assign_coords(lev=ds_sub["lev"])
    )
    p_ip1 = (
        P_interface.isel(ilev=slice(1, None))
        .rename({"ilev": "lev"})
        .assign_coords(lev=ds_sub["lev"])
    )

    p_layer_top = xr.where(p_i < p_ip1, p_i, p_ip1)
    p_layer_bot = xr.where(p_i > p_ip1, p_i, p_ip1)

    pT = p_top_hpa * 100.0
    pB = p_bot_hpa * 100.0

    upper = xr.where(p_layer_top > pT, p_layer_top, pT)
    lower = xr.where(p_layer_bot < pB, p_layer_bot, pB)
    overlap = xr.where(lower > upper, lower - upper, 0.0)

    return (ds_sub["O3"] * overlap * factor).sum(dim="lev")


def calc_O3_pc_30_70_pcap(ds, var="O3", p_top_hpa=30, p_bot_hpa=70):
    """
    O3 -> 30-70 hPa partial column -> 60-90N cosine-weighted mean
    Returns: O3_pc_30_70_60_90N(time)
    """
    ds_cap = ds.sel(lat=slice(60, 90))
    weights = np.cos(np.deg2rad(ds_cap["lat"]))

    da = calc_parc_o3_hybrid(ds_cap, p_top_hpa, p_bot_hpa).weighted(weights).mean(dim=["lat", "lon"])
    da.name = "O3_pc_30_70_60_90N"
    da.attrs["units"] = "DU"
    return da


# ============================================================
# Unified "read + process + save" entry
# ============================================================
def get_var_io_cfg(exp, var):
    cfg = EXP_CFG[exp]

    if var == "U":
        return cfg["U_dir"], cfg["U_pattern"], os.path.join(DATA_ROOT, exp, f"U_60N_{exp}.nc")
    elif var == "T":
        return cfg["T_dir"], cfg["T_pattern"], os.path.join(DATA_ROOT, exp, f"T_min_60_90N_{exp}.nc")
    elif var == "O3":
        return cfg["O3_dir"], cfg["O3_pattern"], os.path.join(DATA_ROOT, exp, f"O3_pc_30_70_60_90N_{exp}.nc")
    else:
        raise ValueError(f"Unsupported var={var}. Only U / T / O3 are supported.")


def process_var_from_ds(ds, var):
    if var == "U":
        da = calc_U60N(ds, var="U")
        da = da.transpose("time", "lev")
        da.name = "U_60N"
        da.attrs["units"] = "m/s"
        da["lev"].attrs["units"] = "Pa"
        return da

    elif var == "T":
        da = calc_minT_pcap(ds, var="T")
        da = da.transpose("time", "lev")
        da.name = "T_min_60_90N"
        da.attrs["units"] = "K"
        da["lev"].attrs["units"] = "Pa"
        return da

    elif var == "O3":
        da = calc_O3_pc_30_70_pcap(ds, var="O3", p_top_hpa=O3_PTOP, p_bot_hpa=O3_PBOT)
        da.name = "O3_pc_30_70_60_90N"
        da.attrs["units"] = "DU"
        return da

    else:
        raise ValueError(f"Unsupported var={var}")


def prepare_and_save_variable(exp, var, force=False):
    """
    Unified external API:
      - var='U' -> save U_60N(time, lev)
      - var='T' -> save T_min_60_90N(time, lev)
      - var='O3' -> save O3_pc_30_70_60_90N(time)
    """
    os.makedirs(os.path.join(DATA_ROOT, exp), exist_ok=True)

    data_dir, pattern, out_nc = get_var_io_cfg(exp, var)

    if os.path.exists(out_nc) and not force:
        print(f"[{exp} {var}] Cache exists, skip: {out_nc}")
        return out_nc

    chunks = {"time": 365}
    if var in ["U", "T"]:
        chunks = {"time": 365, "lat": 48}

    ds = open_daily_files(exp, data_dir, pattern, chunks=chunks)
    print(f"[{exp} {var}] Processing ...")

    da_op = process_var_from_ds(ds, var)

    with ProgressBar():
        da = da_op.compute()

    da.to_netcdf(out_nc)

    year_min = int(da.time.dt.year.min().values)
    year_max = int(da.time.dt.year.max().values)

    if "lev" in da.coords:
        lev_min = float(da.lev.min().values / 100.0)
        lev_max = float(da.lev.max().values / 100.0)
        print(f"[{exp} {var}] saved -> {out_nc}")
        print(f"[{exp} {var}] years: {year_min}-{year_max}, ntime={da.sizes['time']}, lev={lev_min:.2f}-{lev_max:.2f} hPa")
    else:
        print(f"[{exp} {var}] saved -> {out_nc}")
        print(f"[{exp} {var}] years: {year_min}-{year_max}, ntime={da.sizes['time']}")

    ds.close()
    return out_nc

In [ ]:
# ============================================================
# Block 2: Compute pooled low-O3 ranking from cached O3
# ============================================================

def compute_spring_o3_min_5d(da_o3):
    """
    5-day running mean -> select Mar-Apr -> yearly minimum
    Keep only years with >=58 valid spring days.
    """
    da_5d = da_o3.rolling(time=5, center=True, min_periods=5).mean()
    spring = da_5d.where(da_5d.time.dt.month.isin([3, 4]), drop=True)

    day_counts = spring.groupby("time.year").count()
    valid_years = day_counts.year.where(day_counts >= 58, drop=True)

    spring_min = spring.groupby("time.year").min().sel(year=valid_years)
    return spring_min


def build_pooled_o3_ranking(top_n=TOP_N, low_frac=LOW_FRAC):
    """
    Read cached O3 partial column and build pooled ranking:
      - full ranking
      - topN
      - low25%
    """
    rows = []

    for exp in ["BWCN", "B2000WCN"]:
        o3_nc = os.path.join(DATA_ROOT, exp, f"O3_pc_30_70_60_90N_{exp}.nc")
        if not os.path.exists(o3_nc):
            raise FileNotFoundError(f"Missing O3 cache: {o3_nc}")

        da = xr.open_dataarray(o3_nc)
        spring_min = compute_spring_o3_min_5d(da)

        df = pd.DataFrame({
            "exp": exp,
            "year": spring_min["year"].values.astype(int),
            "o3_min": spring_min.values.astype(float),
        })
        rows.append(df)

        print(f"[{exp}] valid spring years = {len(df)}")

    rank_df = pd.concat(rows, ignore_index=True)
    rank_df = rank_df.sort_values("o3_min", ascending=True).reset_index(drop=True)
    rank_df["rank"] = np.arange(1, len(rank_df) + 1)

    n_low = max(1, int(np.floor(low_frac * len(rank_df))))
    top_df = rank_df.head(top_n).copy()
    low_df = rank_df.head(n_low).copy()

    full_csv = os.path.join(RANK_ROOT, "pooled_O3_rank_30_70hPa.csv")
    top_csv  = os.path.join(RANK_ROOT, f"pooled_O3_top{top_n}_30_70hPa.csv")
    low_csv  = os.path.join(RANK_ROOT, "pooled_O3_low25pct_30_70hPa.csv")

    rank_df.to_csv(full_csv, index=False)
    top_df.to_csv(top_csv, index=False)
    low_df.to_csv(low_csv, index=False)

    print("\n[Pooled O3 ranking] Top entries:")
    print(top_df.to_string(index=False))
    print(f"\nSaved full ranking : {full_csv}")
    print(f"Saved top{top_n}     : {top_csv}")
    print(f"Saved low25%     : {low_csv}")

    return rank_df

In [ ]:
# ============================================================
# Block 3: Standard pooled line plotting (JUL(Y-1) -> JUN(Y))
# ============================================================

def get_window_month_order(start_month=7):
    """
    start_month=7 -> [7,8,9,10,11,12,1,2,3,4,5,6]
    """
    return list(range(start_month, 13)) + list(range(1, start_month))


def get_window_month_labels(start_month=7):
    month_names = {
        1: "Jan", 2: "Feb", 3: "Mar", 4: "Apr", 5: "May", 6: "Jun",
        7: "Jul", 8: "Aug", 9: "Sep", 10: "Oct", 11: "Nov", 12: "Dec"
    }
    return [month_names[m] for m in get_window_month_order(start_month)]


def build_shifted_line(da_1d, year, start_month=7):
    """
    Build one cross-year line:
      start_month(Y-1) -> month(start_month-1)(Y)

    Default:
      JUL(Y-1) -> JUN(Y)
    """
    da_1d = da_1d.sortby("time")

    yy = da_1d.time.dt.year.values.astype(int)
    mm = da_1d.time.dt.month.values.astype(int)

    mask = ((yy == (year - 1)) & (mm >= start_month)) | ((yy == year) & (mm < start_month))
    idx = np.where(mask)[0]

    if idx.size == 0:
        return None, None

    seg = da_1d.isel(time=idx).sortby("time")
    seg_months = seg.time.dt.month.values.astype(int)

    month_order = get_window_month_order(start_month)
    month_counts = [int(np.sum(seg_months == m)) for m in month_order]

    if np.any(np.array(month_counts) == 0):
        return None, None

    unique_months_in_order = []
    for m in seg_months:
        if len(unique_months_in_order) == 0 or int(m) != unique_months_in_order[-1]:
            unique_months_in_order.append(int(m))

    if unique_months_in_order != month_order:
        return None, None

    values = seg.values.astype(float)
    meta = {
        "year": int(year),
        "n_days": int(len(values)),
        "month_counts": month_counts,
    }
    return values, meta


def scan_valid_windows(series_map, start_month=7):
    """
    Infer dominant valid window length across experiments.
    Works for noleap / 360_day calendars.
    """
    scan_rows = []

    for exp, da in series_map.items():
        years = np.unique(da.time.dt.year.values.astype(int))
        for y in years:
            _, meta = build_shifted_line(da, int(y), start_month=start_month)
            if meta is None:
                continue

            scan_rows.append({
                "exp": exp,
                "year": int(y),
                "n_days": int(meta["n_days"]),
                "month_counts": meta["month_counts"],
            })

    if len(scan_rows) == 0:
        raise RuntimeError("No valid cross-year windows found.")

    scan_df = pd.DataFrame(scan_rows)

    counts = Counter(scan_df["n_days"].tolist())
    target_n_days = counts.most_common(1)[0][0]

    ref_row = scan_df[scan_df["n_days"] == target_n_days].iloc[0]
    ref_month_counts = list(ref_row["month_counts"])

    return scan_df, target_n_days, ref_month_counts


def make_xticks_from_month_counts(month_counts):
    xticks = [0]
    cum = 0
    for c in month_counts[:-1]:
        cum += int(c)
        xticks.append(cum)
    return xticks


def materialize_lines_from_rows(series_map, rows_df, target_n_days, start_month=7):
    """
    Build pooled lines from exp-year rows, keeping only lines with target_n_days.
    """
    lines = []
    kept = []

    for _, row in rows_df.iterrows():
        exp = str(row["exp"])
        year = int(row["year"])

        if exp not in series_map:
            continue

        vals, meta = build_shifted_line(series_map[exp], year, start_month=start_month)
        if vals is None:
            continue

        if meta["n_days"] != target_n_days:
            continue

        lines.append(vals)
        kept.append({"exp": exp, "year": year})

    if len(lines) == 0:
        return np.empty((0, target_n_days), dtype=float), pd.DataFrame(columns=["exp", "year"])

    return np.vstack(lines), pd.DataFrame(kept)


def build_all_year_rows(series_map):
    rows = []
    for exp, da in series_map.items():
        years = np.unique(da.time.dt.year.values.astype(int))
        for y in years:
            rows.append({"exp": exp, "year": int(y)})
    return pd.DataFrame(rows)


def plot_pooled_line_from_series_map(
    series_map,
    ranking_df,
    out_png,
    ylabel,
    title,
    top_n=TOP_N,
    low_frac=LOW_FRAC,
    ylim=None,
    add_zero_line=False,
    save_pdf=True,
    start_month=7,
):
    """
    Standard pooled line plot:
      - pooled all-year ±1σ
      - pooled low-O3 ±1σ
      - pooled top-N special years

    T plots automatically include the Cl activation threshold line.
    """
    month_labels = get_window_month_labels(start_month)

    # infer valid window length
    _, target_n_days, ref_month_counts = scan_valid_windows(series_map, start_month=start_month)
    xticks = make_xticks_from_month_counts(ref_month_counts)

    # all-year
    all_rows = build_all_year_rows(series_map)
    all_lines, _ = materialize_lines_from_rows(
        series_map, all_rows, target_n_days, start_month=start_month
    )
    if len(all_lines) == 0:
        raise RuntimeError("No valid all-year windows found for plotting.")

    all_mean = np.nanmean(all_lines, axis=0)
    all_std = np.nanstd(all_lines, axis=0)
    '''
    # low-O3
    n_low = max(1, int(np.floor(low_frac * len(ranking_df))))
    low_rows = ranking_df.sort_values("rank").head(n_low)[["exp", "year"]]

    low_lines, _ = materialize_lines_from_rows(
        series_map, low_rows, target_n_days, start_month=start_month
    )
    if len(low_lines) == 0:
        raise RuntimeError("No valid low-O3 windows found for plotting.")

    low_mean = np.nanmean(low_lines, axis=0)
    low_std = np.nanstd(low_lines, axis=0)
    '''
    # top-N
    ranked_rows = ranking_df.sort_values("rank")[["exp", "year"]]
    ranked_lines_valid, ranked_kept = materialize_lines_from_rows(
        series_map, ranked_rows, target_n_days, start_month=start_month
    )
    if len(ranked_lines_valid) == 0:
        raise RuntimeError("No valid ranked windows found for plotting.")

    top_lines = ranked_lines_valid[:top_n]
    top_kept = ranked_kept.iloc[:top_n].reset_index(drop=True)

    print("\nSelected special years:")
    print(top_kept.to_string(index=False))

    # plot
    fig, ax = plt.subplots(figsize=(6.8, 4.2), constrained_layout=True)
    x = np.arange(target_n_days)

    # pooled all-year
    ax.fill_between(
        x, all_mean - all_std, all_mean + all_std,
        color="0.85", alpha=0.95, zorder=0
    )
    ax.plot(x, all_mean, color="black", ls="--", lw=1.8, zorder=2)
    '''
    # pooled low-O3
    ax.fill_between(
        x, low_mean - low_std, low_mean + low_std,
        facecolor="none", edgecolor="0.5", hatch="///", zorder=1
    )
    ax.plot(x, low_mean, color="black", ls="-", lw=2.0, zorder=3)
    '''
    # special years
    year_handles = []
    for i in range(len(top_lines)):
        exp = str(top_kept.loc[i, "exp"])
        year = int(top_kept.loc[i, "year"])
        label = f"{exp}-{year:04d}"
        color = TOP_COLORS[i % len(TOP_COLORS)]

        ax.plot(x, top_lines[i], color=color, lw=1.6, zorder=10)
        year_handles.append(Line2D([0], [0], color=color, lw=1.6, label=label))

    # axes
    ax.set_xlim(0, target_n_days - 1)
    ax.set_xticks(xticks)
    ax.set_xticklabels(month_labels)
    ax.set_ylabel(ylabel)
    ax.set_title(title)

    if add_zero_line:
        ax.axhline(0, color="black", lw=0.8, alpha=0.5)

    if ylim is not None:
        ax.set_ylim(*ylim)

    # --- Always add threshold line for T plots ---
    if "Minimum Temperature" in title:
        CL_THRESH = 197.0
        ax.axhline(CL_THRESH, color="royalblue", linestyle="--", linewidth=1.2, zorder=5)

        if ylim is not None:
            y_text = max(ylim[0] + 2.0, CL_THRESH - 2.0)
        else:
            y_text = CL_THRESH - 2.0

        x_text = max(1, int(0.02 * target_n_days))
        ax.text(
            x_text, y_text,
            "Cl activation threshold",
            color="royalblue",
            fontsize=8,
            va="top",
            ha="left"
        )
    '''
    handles = year_handles + [
        Line2D([0], [0], color="black", lw=1.8, ls="--", label="Pooled all-year"),
        Patch(facecolor="0.85", label="Pooled all-year ±1σ"),
        Line2D([0], [0], color="black", lw=2.0, ls="-", label="Pooled low-O3"),
        Patch(facecolor="none", edgecolor="0.5", hatch="///", label="Pooled low-O3 ±1σ"),
    ]
    '''
    handles = year_handles + [
    Line2D([0], [0], color="black", lw=1.8, ls="--", label="Pooled all-year"),
    Patch(facecolor="0.85", label="Pooled all-year ±1σ"),
    ]
    ax.legend(handles=handles, loc="best", frameon=False, fontsize=8, ncol=2)

    os.makedirs(os.path.dirname(out_png), exist_ok=True)
    plt.savefig(out_png, dpi=300, bbox_inches="tight")

    if save_pdf:
        out_pdf = out_png.replace(".png", ".pdf")
        plt.savefig(out_pdf, bbox_inches="tight")

    plt.show()
    plt.close(fig)

    print(f"Saved: {out_png}")


def plot_variable(var, target_levels=None, save_pdf=True, start_month=7):
    """
    Unified plotting wrapper:
      - var='U' or 'T': multi-level plots
      - var='O3': single-level plot
    """
    rank_csv = os.path.join(RANK_ROOT, "pooled_O3_rank_30_70hPa.csv")
    if not os.path.exists(rank_csv):
        raise FileNotFoundError(f"Missing pooled ranking CSV: {rank_csv}")

    ranking_df = pd.read_csv(rank_csv)

    if var == "U":
        plot_dir = os.path.join(PLOT_ROOT, "U")
        os.makedirs(plot_dir, exist_ok=True)

        ylim_map = {
            1.0:   (-40, 100),
            10.0:  (-20,  70),
            50.0:  (-10,  50),
            100.0: ( -5,  35),
        }

        for target_hpa in target_levels:
            series_map = {}
            actuals = {}

            for exp in ["BWCN", "B2000WCN"]:
                nc = os.path.join(DATA_ROOT, exp, f"U_60N_{exp}.nc")
                da_2d = xr.open_dataarray(nc)
                da_1d, actual_hpa = select_level_da(da_2d, target_hpa)
                series_map[exp] = da_1d
                actuals[exp] = actual_hpa

            ref_hpa = int(round(np.mean(list(actuals.values()))))
            out_png = os.path.join(plot_dir, f"POOLED_U_60N_{ref_hpa}hPa_JulJun_top{TOP_N}.png")

            plot_pooled_line_from_series_map(
                series_map=series_map,
                ranking_df=ranking_df,
                out_png=out_png,
                ylabel=f"U at 60°N, ~{ref_hpa} hPa (m/s)",
                title=f"60°N Zonal Mean U | ~{ref_hpa} hPa",
                top_n=TOP_N,
                low_frac=LOW_FRAC,
                ylim=ylim_map.get(float(target_hpa), None),
                add_zero_line=True,
                save_pdf=save_pdf,
                start_month=start_month,
            )

    elif var == "T":
        plot_dir = os.path.join(PLOT_ROOT, "T")
        os.makedirs(plot_dir, exist_ok=True)

        ylim_map = {
            1.0:   (220, 280),
            10.0:  (180, 250),
            50.0:  (180, 240),
            100.0: (180, 240),
        }

        for target_hpa in target_levels:
            series_map = {}
            actuals = {}

            for exp in ["BWCN", "B2000WCN"]:
                nc = os.path.join(DATA_ROOT, exp, f"T_min_60_90N_{exp}.nc")
                da_2d = xr.open_dataarray(nc)
                da_1d, actual_hpa = select_level_da(da_2d, target_hpa)
                series_map[exp] = da_1d
                actuals[exp] = actual_hpa

            ref_hpa = int(round(np.mean(list(actuals.values()))))
            out_png = os.path.join(plot_dir, f"POOLED_Tmin_60_90N_{ref_hpa}hPa_JulJun_top{TOP_N}.png")

            plot_pooled_line_from_series_map(
                series_map=series_map,
                ranking_df=ranking_df,
                out_png=out_png,
                ylabel=f"Min T at 60–90°N, ~{ref_hpa} hPa (K)",
                title=f"60–90°N Minimum Temperature | ~{ref_hpa} hPa",
                top_n=TOP_N,
                low_frac=LOW_FRAC,
                ylim=ylim_map.get(float(target_hpa), None),
                add_zero_line=False,
                save_pdf=save_pdf,
                start_month=start_month,
            )

    elif var == "O3":
        plot_dir = os.path.join(PLOT_ROOT, "O3")
        os.makedirs(plot_dir, exist_ok=True)

        series_map = {}
        for exp in ["BWCN", "B2000WCN"]:
            nc = os.path.join(DATA_ROOT, exp, f"O3_pc_30_70_60_90N_{exp}.nc")
            da_1d = xr.open_dataarray(nc)
            series_map[exp] = da_1d

        out_png = os.path.join(plot_dir, f"POOLED_O3_30_70hPa_JulJun_top{TOP_N}.png")

        plot_pooled_line_from_series_map(
            series_map=series_map,
            ranking_df=ranking_df,
            out_png=out_png,
            ylabel="O$_3$ partial column, 30–70 hPa (DU)",
            title="60–90°N O$_3$ Partial Column | 30–70 hPa",
            top_n=TOP_N,
            low_frac=LOW_FRAC,
            ylim=None,
            add_zero_line=False,
            save_pdf=save_pdf,
            start_month=start_month,
        )

    else:
        raise ValueError("plot_variable only supports 'U', 'T', or 'O3'")

In [ ]:
# ============================================================
# Block 4: Run all calculations and save caches
# ============================================================

FORCE_REBUILD = False

# 1) O3 cache
for exp in ["BWCN", "B2000WCN"]:
    prepare_and_save_variable(exp, "O3", force=FORCE_REBUILD)

# 2) pooled low-O3 ranking
rank_df = build_pooled_o3_ranking(top_n=TOP_N, low_frac=LOW_FRAC)

# 3) U and T caches
for exp in ["BWCN", "B2000WCN"]:
    prepare_and_save_variable(exp, "U", force=FORCE_REBUILD)
    prepare_and_save_variable(exp, "T", force=FORCE_REBUILD)

In [ ]:
# ============================================================
# Block 5: Run plotting
# ============================================================

TARGET_LEVELS_HPA = [1.0, 10.0, 50.0, 100.0]
TARGET_LEVELS_T_HPA = [10.0, 50.0, 100.0]
SAVE_PDF = True
WINDOW_START_MONTH = 7   # JUL(Y-1) -> JUN(Y)

# U
plot_variable("U", target_levels=TARGET_LEVELS_HPA, save_pdf=SAVE_PDF, start_month=WINDOW_START_MONTH)

# T
plot_variable("T", target_levels=TARGET_LEVELS_T_HPA, save_pdf=SAVE_PDF, start_month=WINDOW_START_MONTH)

# O3
plot_variable("O3", save_pdf=SAVE_PDF, start_month=WINDOW_START_MONTH)

In [ ]:
# ============================================================
# Block 6: Robust reanalysis readers (file-by-file)
# ERA5: U only
# MERRA2: U + T
# Special years are manually specified.
# ============================================================
import os
import glob
import numpy as np
import pandas as pd
import xarray as xr

# ---------------- Paths ----------------
REAN_ROOT = os.path.join(DATA_ROOT, "REANALYSIS")
os.makedirs(REAN_ROOT, exist_ok=True)

ERA5_ROOT = "/pool/datos/reanalisis/era5/6hourly/global"
MERRA2_DAILY_ROOT = "/pool/datos/reanalisis/merra2/daily/zm"

REAN_TARGET_LEVELS_HPA = [1.0, 10.0, 50.0, 100.0]
REAN_WINDOW_START_MONTH = 7   # JUL(Y-1) -> JUN(Y)
REAN_SAVE_PDF = True

# 手动指定事件年（对应窗口后一年）
MANUAL_SPECIAL_YEARS = [1997, 2011, 2020]


# ============================================================
# Generic helpers
# ============================================================
def _rename_common(da_or_ds):
    rename_map = {}

    # time
    if "valid_time" in da_or_ds.coords or "valid_time" in da_or_ds.dims:
        rename_map["valid_time"] = "time"

    # horizontal
    if "latitude" in da_or_ds.coords or "latitude" in da_or_ds.dims:
        rename_map["latitude"] = "lat"
    if "longitude" in da_or_ds.coords or "longitude" in da_or_ds.dims:
        rename_map["longitude"] = "lon"

    # vertical
    if "pressure_level" in da_or_ds.coords or "pressure_level" in da_or_ds.dims:
        rename_map["pressure_level"] = "lev"
    elif "pressure" in da_or_ds.coords or "pressure" in da_or_ds.dims:
        rename_map["pressure"] = "lev"
    elif "level" in da_or_ds.coords or "level" in da_or_ds.dims:
        rename_map["level"] = "lev"

    if rename_map:
        da_or_ds = da_or_ds.rename(rename_map)

    return da_or_ds

def _normalize_lev_to_pa(da):
    if "lev" not in da.coords and "lev" not in da.dims:
        return da

    lev_vals = da["lev"].values
    lev_pa = lev_vals * 100.0 if float(np.nanmax(lev_vals)) <= 2000.0 else lev_vals
    da = da.assign_coords(lev=("lev", lev_pa))
    da["lev"].attrs["units"] = "Pa"
    return da


def _sort_and_unique_time(da):
    da = da.sortby("time")
    tvals = pd.to_datetime(da["time"].values)
    _, keep_idx = np.unique(tvals, return_index=True)
    keep_idx = np.sort(keep_idx)
    da = da.isel(time=keep_idx)
    return da


def _daily_mean(da):
    da = _sort_and_unique_time(da)
    return da.resample(time="1D").mean()


def _concat_series(parts):
    if len(parts) == 0:
        raise RuntimeError("No valid parts to concatenate.")
    da = xr.concat(parts, dim="time")
    da = _sort_and_unique_time(da)
    return da


def _select_target_level(da, target_hpa):
    """
    da must already have lev in Pa.
    Returns selected DataArray (lev removed) and actual_hpa.
    """
    if "lev" not in da.coords and "lev" not in da.dims:
        raise ValueError("No lev coordinate found for level selection")

    target_pa = target_hpa * 100.0
    idx = int(np.argmin(np.abs(da["lev"].values - target_pa)))
    actual_hpa = float(da["lev"].values[idx] / 100.0)
    da = da.isel(lev=idx, drop=True)
    return da, actual_hpa


def build_manual_special_years_df(dataset, special_years=MANUAL_SPECIAL_YEARS):
    """
    Build a ranking-like DataFrame compatible with plot_pooled_line_from_series_map.
    """
    df = pd.DataFrame({
        "exp": [dataset] * len(special_years),
        "year": [int(y) for y in special_years],
        "o3_min": np.arange(len(special_years), dtype=float),  # placeholder only
        "rank": np.arange(1, len(special_years) + 1),
    })
    return df


# ============================================================
# ERA5: U only (yearly 4-daily files)
# ============================================================
import re

def find_era5_u_files():
    """
    Only keep standard pressure-level yearly files:
      u_4daily_YYYYMMDD-YYYYMMDD.nc
    This excludes files like:
      u_4daily_ml_....
      u_theta_....
    """
    all_nc = sorted(glob.glob(os.path.join(ERA5_ROOT, "*.nc")))
    pat = re.compile(r"^u_4daily_\d{8}-\d{8}\.nc$", re.IGNORECASE)

    files = [fp for fp in all_nc if pat.match(os.path.basename(fp))]
    return files


def read_one_era5_u_file(fp, target_hpa):
    """
    Process one ERA5 yearly file:
      u(time, level, latitude, longitude)
      -> select level
      -> 4-daily to daily
      -> zonal mean
      -> 60N
      -> return 1D daily DataArray(time)
    """
    ds = xr.open_dataset(fp)
    ds = _rename_common(ds)

    vname = "u" if "u" in ds.data_vars else "U"
    da = ds[vname]
    da = _rename_common(da)
    da = _normalize_lev_to_pa(da)

    da, actual_hpa = _select_target_level(da, target_hpa)

    # 4-daily -> daily
    da = _daily_mean(da)

    # zonal mean + 60N
    da = da.mean("lon")
    actual_lat = float(da["lat"].sel(lat=60.0, method="nearest").values)
    da = da.sel(lat=60.0, method="nearest")

    da = _sort_and_unique_time(da)
    da.name = "U_60N"
    da.attrs["units"] = "m/s"

    # tiny log
    t0 = pd.to_datetime(da.time.values[0]).strftime("%Y-%m-%d")
    t1 = pd.to_datetime(da.time.values[-1]).strftime("%Y-%m-%d")
    print(f"[ERA5 U] {os.path.basename(fp)} | level={actual_hpa:.2f} hPa | lat={actual_lat:.2f} | {t0} -> {t1}")

    ds.close()
    return da


def read_era5_u_processed(target_hpa):
    files = find_era5_u_files()
    if len(files) == 0:
        raise FileNotFoundError(f"No ERA5 U files found in {ERA5_ROOT}")

    print(f"[ERA5 U] Found {len(files)} yearly files")
    parts = [read_one_era5_u_file(fp, target_hpa) for fp in files]
    da = _concat_series(parts)
    da.name = "U_60N"
    da.attrs["units"] = "m/s"
    return da


# ============================================================
# MERRA2 daily zonal-mean files (monthly)
# Expected filenames:
#   MERRA2_day_u_zm_YYYYMM.nc
#   MERRA2_day_t_zm_YYYYMM.nc
# ============================================================
def find_merra2_daily_files(var):
    if var == "U":
        patterns = ["MERRA2_day_u_zm_*.nc", "MERRA2_day_U_zm_*.nc"]
    elif var == "T":
        patterns = ["MERRA2_day_t_zm_*.nc", "MERRA2_day_T_zm_*.nc"]
    else:
        raise ValueError("MERRA2 only supports U/T")

    files = []
    for pat in patterns:
        files.extend(glob.glob(os.path.join(MERRA2_DAILY_ROOT, pat)))
    return sorted(list(set(files)))


def read_one_merra2_daily_file(fp, var, target_hpa):
    """
    Robustly process one MERRA2 daily zonal-mean monthly file.
    Expected common shapes:
      var(time, pressure/lev, latitude/lat)
    """
    ds = xr.open_dataset(fp)
    ds = _rename_common(ds)

    if var == "U":
        vname = "u" if "u" in ds.data_vars else "U"
    elif var == "T":
        vname = "t" if "t" in ds.data_vars else "T"
    else:
        raise ValueError("MERRA2 only supports U/T")

    da = ds[vname]
    da = _rename_common(da)
    da = _normalize_lev_to_pa(da)

    da, actual_hpa = _select_target_level(da, target_hpa)

    if var == "U":
        actual_lat = float(da["lat"].sel(lat=60.0, method="nearest").values)
        da = da.sel(lat=60.0, method="nearest")
        da.name = "U_60N"
        da.attrs["units"] = "m/s"
        extra = f"lat={actual_lat:.2f}"
    else:
        da = da.sel(lat=slice(60, 90)).min("lat")
        da.name = "T_min_60_90N"
        da.attrs["units"] = "K"
        extra = "lat=60-90N min"

    da = _sort_and_unique_time(da)

    t0 = pd.to_datetime(da.time.values[0]).strftime("%Y-%m-%d")
    t1 = pd.to_datetime(da.time.values[-1]).strftime("%Y-%m-%d")
    print(f"[MERRA2 {var}] {os.path.basename(fp)} | level={actual_hpa:.2f} hPa | {extra} | {t0} -> {t1}")

    ds.close()
    return da


def read_merra2_daily_processed(var, target_hpa):
    files = find_merra2_daily_files(var)
    if len(files) == 0:
        raise FileNotFoundError(f"No MERRA2 {var} daily files found in {MERRA2_DAILY_ROOT}")

    print(f"[MERRA2 {var}] Found {len(files)} monthly files")
    parts = [read_one_merra2_daily_file(fp, var, target_hpa) for fp in files]
    da = _concat_series(parts)

    if var == "U":
        da.name = "U_60N"
        da.attrs["units"] = "m/s"
    else:
        da.name = "T_min_60_90N"
        da.attrs["units"] = "K"

    return da


# ============================================================
# Unified save function
# ============================================================
def save_reanalysis_ut_series(dataset, var, target_hpa, force=False):
    """
    dataset = 'ERA5' or 'MERRA2'
    var     = 'U' or 'T'
    """
    out_dir = os.path.join(REAN_ROOT, dataset)
    os.makedirs(out_dir, exist_ok=True)

    level_int = int(round(target_hpa))

    if var == "U":
        out_nc = os.path.join(out_dir, f"{dataset}_U_60N_{level_int}hPa_daily.nc")
    elif var == "T":
        out_nc = os.path.join(out_dir, f"{dataset}_Tmin_60_90N_{level_int}hPa_daily.nc")
    else:
        raise ValueError("save_reanalysis_ut_series only supports U/T")

    if os.path.exists(out_nc) and not force:
        print(f"[{dataset} {var} {level_int}hPa] Cache exists, skip: {out_nc}")
        return out_nc

    if dataset == "ERA5":
        if var != "U":
            raise ValueError("ERA5 currently only supports U.")
        da = read_era5_u_processed(target_hpa)

    elif dataset == "MERRA2":
        da = read_merra2_daily_processed(var, target_hpa)

    else:
        raise ValueError("dataset must be ERA5 or MERRA2")

    da.to_netcdf(out_nc)

    years = np.unique(da.time.dt.year.values.astype(int))
    print(f"Saved: {out_nc}")
    print(f"[{dataset} {var}] years: {years.min()}-{years.max()}, ntime={da.sizes['time']}")

    return out_nc

In [ ]:
# ============================================================
# Block 7: Build reanalysis caches
# ERA5: U only
# MERRA2: U + T
# ============================================================

REAN_FORCE_REBUILD = False

# ERA5: U only
for lev in REAN_TARGET_LEVELS_HPA:
    save_reanalysis_ut_series("ERA5", "U", target_hpa=lev, force=REAN_FORCE_REBUILD)

# MERRA2: U + T
for lev in REAN_TARGET_LEVELS_HPA:
    save_reanalysis_ut_series("MERRA2", "U", target_hpa=lev, force=REAN_FORCE_REBUILD)
    save_reanalysis_ut_series("MERRA2", "T", target_hpa=lev, force=REAN_FORCE_REBUILD)

In [ ]:
import xarray as xr
import glob
import os

# 你的数据根目录（从日志中提取）
rean_cache_path = "/home/weiji/restart_exam/code/20260226B2000WCN_BWCN_MERRA2_COMPARE/RESULT/data/REANALYSIS"

# 查找该目录下所有的 .nc 文件
nc_files = sorted(glob.glob(os.path.join(rean_cache_path, "**/*.nc"), recursive=True))

print(f"{'File Name':<45} | {'Start':<12} | {'End':<12} | {'Days':<6}")
print("-" * 85)

for f in nc_files:
    try:
        with xr.open_dataset(f) as ds:
            t_min = ds.time.dt.strftime('%Y-%m-%d').values[0]
            t_max = ds.time.dt.strftime('%Y-%m-%d').values[-1]
            n_days = len(ds.time)
            fname = os.path.basename(f)
            print(f"{fname:<45} | {t_min:<12} | {t_max:<12} | {n_days:<6}")
    except Exception as e:
        print(f"Error reading {f}: {e}")

In [ ]:
# ============================================================
# Block 8: Plot reanalysis with MANUAL special years
# - Uses existing plot_pooled_line_from_series_map(...)
# - Keeps fixed year-color correspondence across datasets:
#     1997 -> color #1
#     2011 -> color #2
#     2020 -> color #3
# - Fixes leap-year issue by monkey-patching the window builder
#   to accept 366-day JUL->JUN windows and drop Feb 29.
# ============================================================

REAN_SAVE_PDF = True


# ------------------------------------------------------------
# 1) Leap-safe window builder (365-day standardized)
# ------------------------------------------------------------
def build_shifted_line_fixed365(da_1d, year, start_month=7):
    """
    Compatible replacement for the original build_shifted_line(...).

    Returns:
      (line, meta)

    where:
      - line is a 365-day numpy array, or None
      - meta is a dict, or None

    Rules:
      - accept both 365-day and 366-day JUL(Y-1)->JUN(Y) windows
      - if 366 days, drop Feb 29
      - require all 12 months present and in correct order
    """
    da_1d = da_1d.sortby("time")
    t = pd.to_datetime(da_1d.time.values)

    yy = t.year
    mm = t.month

    month_order = list(range(start_month, 13)) + list(range(1, start_month))

    mask = ((yy == (year - 1)) & (mm >= start_month)) | ((yy == year) & (mm < start_month))
    idx = np.where(mask)[0]

    if idx.size == 0:
        return None, None

    seg = da_1d.isel(time=idx).sortby("time")
    seg_t = pd.to_datetime(seg.time.values)
    seg_months = seg_t.month.values

    month_counts = [int(np.sum(seg_months == m)) for m in month_order]
    if any(c == 0 for c in month_counts):
        return None, None

    unique_months_seen = []
    for m in seg_months:
        if len(unique_months_seen) == 0 or int(m) != unique_months_seen[-1]:
            unique_months_seen.append(int(m))
    if unique_months_seen != month_order:
        return None, None

    vals = np.asarray(seg.values, dtype=float)

    # leap year window: remove Feb 29
    if len(vals) == 366:
        keep = ~((seg_t.month == 2) & (seg_t.day == 29))
        vals = vals[keep]
        seg_t = seg_t[keep]

        # recompute month counts after removing Feb 29
        seg_months = seg_t.month.values
        month_counts = [int(np.sum(seg_months == m)) for m in month_order]

    if len(vals) != 365:
        return None, None

    meta = {
        "n_days": int(len(vals)),
        "month_counts": month_counts,
        "first_time": str(seg_t[0]),
        "last_time": str(seg_t[-1]),
        "year": int(year),
    }

    return vals, meta

# ------------------------------------------------------------
# 2) Monkey-patch existing plotting helpers to use fixed365
# ------------------------------------------------------------
def _patch_existing_window_builders():
    """
    Replace whichever existing window-builder function is present
    so that plot_pooled_line_from_series_map(...) will include leap windows.
    """
    patched = []

    if "build_shifted_line" in globals():
        globals()["build_shifted_line"] = build_shifted_line_fixed365
        patched.append("build_shifted_line")

    if "build_octsep_line" in globals():
        globals()["build_octsep_line"] = build_shifted_line_fixed365
        patched.append("build_octsep_line")

    # If your existing materialize function calls a builder internally by name,
    # the above is usually enough. We just print what got patched.
    if len(patched) == 0:
        print("[Block8] Warning: no known window-builder function found to patch.")
        print("[Block8] If ERA5 2020 still does not appear, your base plotting code may use a differently named builder.")
    else:
        print(f"[Block8] Patched window builder(s): {patched}")


_patch_existing_window_builders()


# ------------------------------------------------------------
# 3) Filter only valid manual special years for a dataset
# ------------------------------------------------------------
def filter_valid_special_years_for_dataset(da_1d, candidate_years, start_month=7):
    """
    Keep only years with a complete shifted window.
    Order is preserved exactly as in candidate_years.
    """
    valid_years = []
    for y in candidate_years:
        line, meta = build_shifted_line_fixed365(da_1d, int(y), start_month=start_month)
        if line is not None and meta is not None:
            valid_years.append(int(y))
    return valid_years

def build_ordered_manual_ranking_df(dataset, ordered_years):
    """
    Build a ranking-like DataFrame compatible with plot_pooled_line_from_series_map.
    IMPORTANT:
      - preserves given year order exactly
      - preserves year-color correspondence across datasets
    """
    return pd.DataFrame({
        "exp": [dataset] * len(ordered_years),
        "year": [int(y) for y in ordered_years],
        "o3_min": np.arange(len(ordered_years), dtype=float),  # placeholder only
        "rank": np.arange(1, len(ordered_years) + 1),
    })


# ------------------------------------------------------------
# 4) Main plotting wrapper
# ------------------------------------------------------------
def plot_reanalysis_ut_manual(dataset, var, target_levels, save_pdf=True, start_month=7):
    if var == "U":
        plot_dir = os.path.join(PLOT_ROOT, "REANALYSIS", dataset, "U")
        os.makedirs(plot_dir, exist_ok=True)

        ylim_map = {
            1.0:   (-40, 100),
            10.0:  (-20,  70),
            50.0:  (-10,  50),
            100.0: ( -5,  35),
        }

        for lev in target_levels:
            level_int = int(round(lev))
            nc = os.path.join(REAN_ROOT, dataset, f"{dataset}_U_60N_{level_int}hPa_daily.nc")
            da = xr.open_dataarray(nc)

            # keep MANUAL_SPECIAL_YEARS order exactly
            valid_years = filter_valid_special_years_for_dataset(
                da, MANUAL_SPECIAL_YEARS, start_month=start_month
            )

            if len(valid_years) == 0:
                print(f"[{dataset} U {level_int}hPa] No valid special years. Skip.")
                continue

            ranking_df = build_ordered_manual_ranking_df(dataset, valid_years)
            print(f"[{dataset} U {level_int}hPa] valid special years: {valid_years}")

            series_map = {dataset: da}
            out_png = os.path.join(
                plot_dir,
                f"{dataset}_U_60N_{level_int}hPa_JulJun_manualTop{len(valid_years)}.png"
            )

            plot_pooled_line_from_series_map(
                series_map=series_map,
                ranking_df=ranking_df,
                out_png=out_png,
                ylabel=f"U at 60°N, ~{level_int} hPa (m/s)",
                title=f"{dataset} | 60°N Zonal Mean U | ~{level_int} hPa",
                top_n=len(valid_years),
                low_frac=1.0,   # use valid manual years as composite
                ylim=ylim_map.get(float(lev), None),
                add_zero_line=True,
                save_pdf=save_pdf,
                start_month=start_month,
            )

    elif var == "T":
        if dataset == "ERA5":
            raise ValueError("ERA5 T is not available in the current archive.")

        plot_dir = os.path.join(PLOT_ROOT, "REANALYSIS", dataset, "T")
        os.makedirs(plot_dir, exist_ok=True)

        ylim_map = {
            1.0:   (220, 280),
            10.0:  (180, 250),
            50.0:  (180, 240),
            100.0: (180, 240),
        }

        for lev in target_levels:
            level_int = int(round(lev))
            nc = os.path.join(REAN_ROOT, dataset, f"{dataset}_Tmin_60_90N_{level_int}hPa_daily.nc")
            da = xr.open_dataarray(nc)

            # keep MANUAL_SPECIAL_YEARS order exactly
            valid_years = filter_valid_special_years_for_dataset(
                da, MANUAL_SPECIAL_YEARS, start_month=start_month
            )

            if len(valid_years) == 0:
                print(f"[{dataset} T {level_int}hPa] No valid special years. Skip.")
                continue

            ranking_df = build_ordered_manual_ranking_df(dataset, valid_years)
            print(f"[{dataset} T {level_int}hPa] valid special years: {valid_years}")

            series_map = {dataset: da}
            out_png = os.path.join(
                plot_dir,
                f"{dataset}_Tmin_60_90N_{level_int}hPa_JulJun_manualTop{len(valid_years)}.png"
            )

            plot_pooled_line_from_series_map(
                series_map=series_map,
                ranking_df=ranking_df,
                out_png=out_png,
                ylabel=f"Min T at 60–90°N, ~{level_int} hPa (K)",
                title=f"{dataset} | 60–90°N Minimum Temperature | ~{level_int} hPa",
                top_n=len(valid_years),
                low_frac=1.0,   # use valid manual years as composite
                ylim=ylim_map.get(float(lev), None),
                add_zero_line=False,
                save_pdf=save_pdf,
                start_month=start_month,
            )

    else:
        raise ValueError("plot_reanalysis_ut_manual only supports U/T")


# ------------------------------------------------------------
# 5) Run plotting
# ------------------------------------------------------------

# ERA5: U only
plot_reanalysis_ut_manual(
    dataset="ERA5",
    var="U",
    target_levels=REAN_TARGET_LEVELS_HPA,
    save_pdf=REAN_SAVE_PDF,
    start_month=REAN_WINDOW_START_MONTH,
)

# MERRA2: U + T
plot_reanalysis_ut_manual(
    dataset="MERRA2",
    var="U",
    target_levels=REAN_TARGET_LEVELS_HPA,
    save_pdf=REAN_SAVE_PDF,
    start_month=REAN_WINDOW_START_MONTH,
)

plot_reanalysis_ut_manual(
    dataset="MERRA2",
    var="T",
    target_levels=REAN_TARGET_LEVELS_HPA,
    save_pdf=REAN_SAVE_PDF,
    start_month=REAN_WINDOW_START_MONTH,
)